In [ ]:
# import tools to work with numbers and files
# import tools to process maps and images
# import shared settings and basic data
import sys
import warnings
from pathlib import Path
import random

import numpy as np
import pandas as pd
import geopandas as gpd
import scipy.sparse as sp
from scipy.sparse.linalg import lgmres, spsolve
import cv2
import rasterio
from rasterio.features import rasterize as rio_rasterize
from rasterio.transform import rowcol
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent))
import shared


In [ ]:
# set folder locations for saving output files
# create output folder if needed
# set step size for walking difficulty calculation
figures_dir = shared.OUT / "figures" / "circuit"
figures_dir.mkdir(parents=True, exist_ok=True)

vector_gis_dir = shared.OUT / "vector_gis"
vector_gis_dir.mkdir(exist_ok=True)

footprints, doors_polygons, doors_points, crosswalk = shared.get_base_preprocessed_data()
dem = shared.load_dem()
doors_points = shared.sample_dem_at_doors(doors_points)

hillshade = shared.hillshade(dem["disp"])
xmin, ymin, xmax, ymax = footprints.total_bounds


In [ ]:
# get step directions and walking distances for surrounding grid cells
def get_offsets():
    return [(0, 1, 1.0), (1, 0, 1.0), (1, 1, np.sqrt(2)), (1, -1, np.sqrt(2))]


In [ ]:
# build connection weights between neighboring grid cells based on travel difficulty
# check four surrounding cells or eight surrounding cells based on setting
# calculate resistance from average walking difficulty and distance
# calculate connection flow strength as inverse of resistance
# block flow completely through buildings and obstacles
# remove invalid or zero connections
# connect isolated points so the math works properly
def build_conductance_matrix(cost_raster, connectivity=8):
    """
    Constructs a sparse conductance (Laplacian) matrix for the grid.
    
    Parameters:
        cost_raster: 2D numpy array of travel costs.
        connectivity: 4 or 8 neighbor connections.
        
    Returns:
        L: scipy.sparse.csr_matrix representing the Laplacian.
    """
    h, w = cost_raster.shape
    n_pixels = h * w
    
    if connectivity == 8:
        offsets = get_offsets()
    else:
        offsets = [
            (0, 1, 1.0),
            (1, 0, 1.0)
        ]
        
    rows = []
    cols = []
    conductances = []
    
    for dr, dc, dist in offsets:
        r_start = 0
        r_end = h - dr
        c_start = max(0, -dc)
        c_end = min(w, w - dc)
        
        if r_start >= r_end or c_start >= c_end:
            continue
            
        r, c = np.meshgrid(np.arange(r_start, r_end), np.arange(c_start, c_end), indexing='ij')
        r = r.ravel()
        c = c.ravel()
        
        tr = r + dr
        tc = c + dc
        
        idx1 = r * w + c
        idx2 = tr * w + tc
        
        cost1 = cost_raster[r, c]
        cost2 = cost_raster[tr, tc]
        
        avg_cost = 0.5 * (cost1 + cost2)
        avg_cost = np.maximum(avg_cost, 1e-6)
        r_val = avg_cost * dist
        g_val = 1.0 / r_val
        
        g_val[(cost1 >= 1e8) | (cost2 >= 1e8)] = 0.0
        
        rows.append(idx1)
        cols.append(idx2)
        conductances.append(g_val)
        
        rows.append(idx2)
        cols.append(idx1)
        conductances.append(g_val)
        
    if len(rows) > 0:
        rows = np.concatenate(rows)
        cols = np.concatenate(cols)
        conductances = np.concatenate(conductances)
        
        valid_edges = conductances > 1e-12
        rows = rows[valid_edges]
        cols = cols[valid_edges]
        conductances = conductances[valid_edges]
    else:
        rows = np.array([], dtype=np.int32)
        cols = np.array([], dtype=np.int32)
        conductances = np.array([], dtype=np.float64)
        
    diag_conductance = np.bincount(rows.astype(int), weights=conductances, minlength=n_pixels)
    
    disconnected = (diag_conductance < 1e-12)
    diag_conductance[disconnected] = 1.0
    
    diag_indices = np.arange(n_pixels)
    all_rows = np.concatenate([rows, diag_indices])
    all_cols = np.concatenate([cols, diag_indices])
    all_vals = np.concatenate([-conductances, diag_conductance])
    
    L = sp.coo_matrix((all_vals, (all_rows, all_cols)), shape=(n_pixels, n_pixels)).tocsr()
    return L


In [ ]:
# move any points inside buildings to the closest open ground cell
def snap_pixels(pixels, building_mask):
    """Snaps pixels inside buildings to the nearest non-building pixel."""
    h, w = building_mask.shape
    snapped_pixels = []
    for r, c in pixels:
        if building_mask[r, c] == 1:
            snapped = False
            for radius in range(1, 15):
                for dr in range(-radius, radius + 1):
                    for dc in range(-radius, radius + 1):
                        if abs(dr) != radius and abs(dc) != radius:
                            continue
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < h and 0 <= nc < w and building_mask[nr, nc] == 0:
                            r, c = nr, nc
                            snapped = True
                            break
                    if snapped:
                        break
                if snapped:
                    break
        snapped_pixels.append((r, c))
    return snapped_pixels


In [ ]:
# calculate electrical potential across every point on the grid
def solve_linear_system(L, I, method='lgmres'):
    """Solves a sparse linear system LV = I with a fallback to direct spsolve."""
    if method == 'lgmres':
        try:
            V, info = lgmres(L, I, tol=1e-5, maxiter=300)
            if info != 0:
                from scipy.sparse.linalg import spsolve
                V = spsolve(L, I)
        except Exception:
            from scipy.sparse.linalg import spsolve
            V = spsolve(L, I)
    else:
        from scipy.sparse.linalg import spsolve
        V = spsolve(L, I)
    return V


In [ ]:
# calculate flow intensity moving through each grid cell
# check horizontal and vertical movement across grid cells
# add up all flow entering and leaving each point
def compute_current_density_from_potentials(V, cost_raster, connectivity=8):
    """Computes node current densities from node potentials."""
    h, w = cost_raster.shape
    n_pixels = h * w
    current_density = np.zeros(n_pixels, dtype=np.float64)
    
    if connectivity == 8:
        offsets = get_offsets()
    else:
        offsets = [
            (0, 1, 1.0),
            (1, 0, 1.0)
        ]
        
    for dr, dc, dist in offsets:
        r_start = 0
        r_end = h - dr
        c_start = max(0, -dc)
        c_end = min(w, w - dc)
        
        if r_start >= r_end or c_start >= c_end:
            continue
            
        r, c = np.meshgrid(np.arange(r_start, r_end), np.arange(c_start, c_end), indexing='ij')
        r = r.ravel()
        c = c.ravel()
        
        tr = r + dr
        tc = c + dc
        
        idx1 = r * w + c
        idx2 = tr * w + tc
        
        cost1 = cost_raster[r, c]
        cost2 = cost_raster[tr, tc]
        avg_cost = 0.5 * (cost1 + cost2)
        avg_cost = np.maximum(avg_cost, 1e-6)
        g_val = 1.0 / (avg_cost * dist)
        g_val[(cost1 >= 1e8) | (cost2 >= 1e8)] = 0.0
        
        v1 = V[idx1]
        v2 = V[idx2]
        current = g_val * (v1 - v2)
        abs_current = np.abs(current)
        
        np.add.at(current_density, idx1, 0.5 * abs_current)
        np.add.at(current_density, idx2, 0.5 * abs_current)
        
    return current_density.reshape((h, w))


In [ ]:
# fix destination cell value to zero so flow moves toward it
def apply_dirichlet_boundary(L, sink_flat):
    L_solve = L.copy()
    start_idx = L_solve.indptr[sink_flat]
    end_idx = L_solve.indptr[sink_flat + 1]
    L_solve.data[start_idx:end_idx] = 0.0
    diag_idx = np.where(L_solve.indices[start_idx:end_idx] == sink_flat)[0]
    if len(diag_idx) > 0:
        L_solve.data[start_idx + diag_idx[0]] = 1.0
    return L_solve


In [ ]:
# simulate flow from all doors across the site to one main target location
# shrink image size to speed up calculation
# adjust door coordinates to match smaller image
# move doors away from building walls to open ground
# pick door closest to center as target location
# build connection weights across the grid
# send equal flow from every door
# set target door value to zero
# solve for flow potentials across the grid
# calculate flow intensity for every cell
# resize flow map back to full image size
# scale flow intensity values between zero and one
def compute_supernode_current(cost_raster, entrance_pixels, shape, sink_idx=None):
    """
    Computes current density where all doors are connected to a single reference/sink door.
    
    Parameters:
        cost_raster: High-res cost surface.
        entrance_pixels: High-res door coordinates.
        shape: Original shape (H, W).
        sink_idx: Index of the door to use as the ground/sink.
    """
    h, w = cost_raster.shape
    max_pixels = 250000
    
    total_pixels = h * w
    if total_pixels > max_pixels:
        scale = np.sqrt(max_pixels / total_pixels)
        h_down = int(round(h * scale))
        w_down = int(round(w * scale))
    else:
        h_down, w_down = h, w
        
    h_down = max(1, h_down)
    w_down = max(1, w_down)
    
    if h_down != h or w_down != w:
        cost_down = cv2.resize(cost_raster, (w_down, h_down), interpolation=cv2.INTER_NEAREST)
    else:
        cost_down = cost_raster.copy()
        
    entrance_pixels_down = []
    for r, c in entrance_pixels:
        r_d = int(round(r * (h_down / h)))
        c_d = int(round(c * (w_down / w)))
        r_d = np.clip(r_d, 0, h_down - 1)
        c_d = np.clip(c_d, 0, w_down - 1)
        entrance_pixels_down.append((r_d, c_d))
        
    building_mask_down = (cost_down > 1e6).astype(np.uint8)
    entrance_pixels_down = snap_pixels(entrance_pixels_down, building_mask_down)
    
    if sink_idx is None:
        r_coords = np.array([p[0] for p in entrance_pixels])
        c_coords = np.array([p[1] for p in entrance_pixels])
        r_mean = np.mean(r_coords)
        c_mean = np.mean(c_coords)
        dists = np.hypot(r_coords - r_mean, c_coords - c_mean)
        sink_idx = np.argmin(dists)
        
    L = build_conductance_matrix(cost_down, connectivity=8)
    
    n_nodes = h_down * w_down
    I = np.zeros(n_nodes, dtype=np.float64)
    
    sink_pixel = entrance_pixels_down[sink_idx]
    sink_flat = sink_pixel[0] * w_down + sink_pixel[1]
    
    for idx, (r, c) in enumerate(entrance_pixels_down):
        flat_idx = r * w_down + c
        if flat_idx != sink_flat:
            I[flat_idx] += 1.0
            
    total_injected = np.sum(I)
    I[sink_flat] = -total_injected
    
    L_solve = apply_dirichlet_boundary(L, sink_flat)
        
    I_solve = I.copy()
    I_solve[sink_flat] = 0.0
    
    V = solve_linear_system(L_solve, I_solve, method='lgmres')
    
    density_down = compute_current_density_from_potentials(V, cost_down, connectivity=8)
    
    density_upsampled = cv2.resize(density_down, (shape[1], shape[0]), interpolation=cv2.INTER_LINEAR)
    
    if density_upsampled.max() > 0:
        density_upsampled = density_upsampled / density_upsampled.max()
        
    return density_upsampled


In [ ]:
# simulate flow between every pair of connected doors
# shrink image size to speed up calculation
# adjust door coordinates to match smaller image
# move doors away from building walls to open ground
# create pairs of doors that connect to each other
# build connection weights across the grid
# set destination door value to zero
# solve for flow potentials across the grid
# calculate flow intensity between this pair of doors
# resize flow map back to full image size
# scale total flow intensity values between zero and one
def compute_pairwise_current(cost_raster, entrance_pixels, max_pairs=50):
    """
    Computes pairwise current density between selected pairs of doors.
    """
    h, w = cost_raster.shape
    max_pixels = 250000
    
    total_pixels = h * w
    if total_pixels > max_pixels:
        scale = np.sqrt(max_pixels / total_pixels)
        h_down = int(round(h * scale))
        w_down = int(round(w * scale))
    else:
        h_down, w_down = h, w
        
    h_down = max(1, h_down)
    w_down = max(1, w_down)
    
    if h_down != h or w_down != w:
        cost_down = cv2.resize(cost_raster, (w_down, h_down), interpolation=cv2.INTER_NEAREST)
    else:
        cost_down = cost_raster.copy()
        
    entrance_pixels_down = []
    for r, c in entrance_pixels:
        r_d = int(round(r * (h_down / h)))
        c_d = int(round(c * (w_down / w)))
        r_d = np.clip(r_d, 0, h_down - 1)
        c_d = np.clip(c_d, 0, w_down - 1)
        entrance_pixels_down.append((r_d, c_d))
        
    building_mask_down = (cost_down > 1e6).astype(np.uint8)
    entrance_pixels_down = snap_pixels(entrance_pixels_down, building_mask_down)
    
    pairs = []
    for i in range(len(entrance_pixels_down)):
        for j in range(i + 1, len(entrance_pixels_down)):
            p1 = entrance_pixels_down[i]
            p2 = entrance_pixels_down[j]
            if p1 != p2:
                pairs.append((p1, p2))
                
    if len(pairs) > max_pairs:
        import random
        random.seed(42)
        pairs = random.sample(pairs, max_pairs)
        
    L = build_conductance_matrix(cost_down, connectivity=8)
    
    accumulated_density = np.zeros((h_down, w_down), dtype=np.float64)
    
    print(f"Solving pairwise current for {len(pairs)} pairs...")
    for idx, (p1, p2) in enumerate(pairs):
        n_nodes = h_down * w_down
        I = np.zeros(n_nodes, dtype=np.float64)
        
        flat_p1 = p1[0] * w_down + p1[1]
        flat_p2 = p2[0] * w_down + p2[1]
        
        I[flat_p1] = 1.0
        I[flat_p2] = -1.0
        
        L_solve = apply_dirichlet_boundary(L, flat_p2)
            
        I_solve = I.copy()
        I_solve[flat_p2] = 0.0
        
        V = solve_linear_system(L_solve, I_solve, method='lgmres')
        
        density_down = compute_current_density_from_potentials(V, cost_down, connectivity=8)
        accumulated_density += density_down
        
        if (idx + 1) % 10 == 0 or idx == len(pairs) - 1:
            print(f"  Processed {idx + 1}/{len(pairs)} pairs")
            
    density_upsampled = cv2.resize(accumulated_density, (w, h), interpolation=cv2.INTER_LINEAR)
    
    if density_upsampled.max() > 0:
        density_upsampled = density_upsampled / density_upsampled.max()
        
    return density_upsampled


In [ ]:
# calculate walking difficulty map from terrain steepness
# load ground elevation data
# calculate terrain steepness in both directions
# use hiking formula to turn steepness into walking speed and difficulty
# block walking through building locations
# save walking difficulty map to file
print("Computing terrain-informed Tobler cost surface ...")
footprints_dem = footprints.to_crs(dem["crs"]) if footprints.crs != dem["crs"] else footprints.copy()

dy, dx = np.gradient(dem["disp"], dem["res"], dem["res"])
slope_mag = np.sqrt(dx**2 + dy**2)

tobler_kmh = 6.0 * np.exp(-3.5 * np.abs(slope_mag + 0.05))
tobler_ms = tobler_kmh / 3.6
cost_tobler = 1.0 / np.maximum(tobler_ms, 1e-6)

shapes = [(geom, 1) for geom in footprints_dem.geometry if geom is not None]
building_mask = rio_rasterize(
    shapes,
    out_shape=dem["arr"].shape,
    transform=dem["transform"],
    fill=0,
    dtype=np.uint8,
)
cost_obs = np.where(building_mask == 1, 1e9, 0.0)
cost_surface = 0.5 * cost_tobler + cost_obs

profile_cost = dem["profile"].copy()
profile_cost.update(dtype="float32", count=1)
with rasterio.open(shared.OUT / "cost_surface_tobler.tif", "w", **profile_cost) as dst:
    dst.write(cost_surface.astype(np.float32), 1)
print("  Cost surface saved -> outputs/cost_surface_tobler.tif")


In [ ]:
# load door coordinates and match them to grid cells
# make sure no doors fall inside building boundaries
# find index of main church door to use as target location
door_coords = [(g.x, g.y) for g in doors_points.geometry]
rows_px, cols_px = rowcol(
    dem["transform"],
    [p[0] for p in door_coords],
    [p[1] for p in door_coords],
)

h, w = dem["arr"].shape
entrance_pixels = []
valid_door_rows = []

for i, (r, c) in enumerate(zip(rows_px, cols_px)):
    if not (0 <= r < h and 0 <= c < w):
        continue
    if building_mask[r, c] == 1:
        snapped = False
        for dr in range(-5, 6):
            for dc in range(-5, 6):
                nr, nc = r + dr, c + dc
                if 0 <= nr < h and 0 <= nc < w and building_mask[nr, nc] == 0:
                    r, c = nr, nc
                    snapped = True
                    break
            if snapped:
                break
    entrance_pixels.append((r, c))
    valid_door_rows.append(doors_points.iloc[i])
    
print(f"  Projected {len(entrance_pixels)} doors to raster pixels.")

sink_idx = None
for idx, row in enumerate(valid_door_rows):
    if str(row["chapel_id"]) == "180":
        sink_idx = idx
        break


In [ ]:
# run flow simulation from all doors to main church
# save main church flow intensity map to file
print("Solving supernode current density (all-to-one, sink at Chapel 180) ...")
supernode_density = compute_supernode_current(
    cost_surface,
    entrance_pixels,
    cost_surface.shape,
    sink_idx=sink_idx
)

profile_density = dem["profile"].copy()
profile_density.update(dtype="float32", count=1)
with rasterio.open(shared.OUT / "circuit_current_supernode.tif", "w", **profile_density) as dst:
    dst.write(supernode_density.astype(np.float32), 1)
print("  Supernode current density saved -> outputs/circuit_current_supernode.tif")


In [ ]:
# run flow simulation between all pairs of doors
# save pairwise flow intensity map to file
print("Solving pairwise current density (max_pairs=50) ...")
pairwise_density = compute_pairwise_current(
    cost_surface,
    entrance_pixels,
    max_pairs=50
)

with rasterio.open(shared.OUT / "circuit_current_pairwise.tif", "w", **profile_density) as dst:
    dst.write(pairwise_density.astype(np.float32), 1)
print("  Pairwise current density saved -> outputs/circuit_current_pairwise.tif")


In [ ]:
# save door locations to file for reference
doors_points.to_crs("EPSG:4326").to_file(
    str(vector_gis_dir / "circuit_door_points.geojson"), driver="GeoJSON"
)
print("  Door points GeoJSON saved -> outputs/circuit_door_points.geojson")


In [ ]:
# create function to draw maps of walking flow across the site
# draw flow map image on screen
# overlay building outlines and door locations
# draw roads and walking paths
# save finished picture to output folder
def plot_density(density, title, filename):
    fp_dem = footprints.to_crs(dem["crs"]) if footprints.crs != dem["crs"] else footprints.copy()
    doors_dem = doors_polygons.to_crs(dem["crs"]) if doors_polygons.crs != dem["crs"] else doors_polygons.copy()

    fig, ax = plt.subplots(figsize=(18, 15))
    ax.imshow(dem["disp"], extent=dem["extent"], origin="upper",
              cmap="terrain", alpha=0.6, vmin=dem["e_min"], vmax=dem["e_max"],
              rasterized=True)
    ax.imshow(hillshade, extent=dem["extent"], origin="upper",
              cmap="gray", alpha=0.3, rasterized=True)
              
    density_masked = np.where(density > 0.005, density, np.nan)
    im = ax.imshow(density_masked, extent=dem["extent"], origin="upper",
                   cmap="inferno", alpha=0.85, vmin=0.0, vmax=1.0, rasterized=True)
                   
    fp_dem.plot(ax=ax, color="none", edgecolor="#333333", linewidth=0.5, alpha=0.7)

    for _, row in doors_dem.iterrows():
        xs, ys = row.geometry.xy
        ax.plot(xs, ys, color=shared.DIR_CLR.get(row["direction"], "#aaa"),
                linewidth=2.0, zorder=5)
                
    shared._draw_chapel_ids(ax, fp_dem, fontsize=4, color="white",
                            bbox=dict(boxstyle="round,pad=0.1", fc="black", alpha=0.4, lw=0))
                            
    plt.colorbar(im, ax=ax, label="Current Density (normalized)", shrink=0.5)
    ax.set_title(title, fontsize=15, fontweight="bold")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.set_xlim(xmin - 50, xmax + 50)
    ax.set_ylim(ymin - 50, ymax + 50)
    plt.tight_layout()

    shared.save_fig(figures_dir, filename)
    plt.show()


In [ ]:
# draw and save map of flow from all doors to main church
plot_density(supernode_density, "El-Bagawat Resistor Network — Supernode Current Density (all → Chapel 180)", "17_circuit_supernode_density.png")


In [ ]:
# draw and save map of flow between all door pairs
plot_density(pairwise_density, "El-Bagawat Resistor Network — Pairwise Current Density (max_pairs=50)", "18_circuit_pairwise_density.png")
